# Major CS — V3 com o histórico de todos os tiers

**Objetivo** — medir se os jogos de tiers menores aumentam a cobertura e melhoram a previsão de séries para equipes com poucas partidas no Tier 1.

O notebook original `EDA.ipynb` registra a retirada da V2 de janela de 90 dias. A referência reproduzível usada aqui é o modelo **V1 + ranking HLTV (simétrico)**, que já foi usado na previsão anterior do Major. A V3 mantém suas cinco entradas, o histórico acumulado, a imputação, a padronização, a regressão logística e os mesmos cortes de datas. A mudança é a base de jogos usada para montar o histórico e treinar.

1. Carregar e auditar os CSVs.
2. Montar o alvo e o histórico anterior de cada equipe.
3. Acrescentar o ranking HLTV e reproduzir a referência.
4. Treinar e comparar as versões nos mesmos confrontos.
5. Salvar o modelo e simular o Major de Singapura.

Execute as células em ordem no kernel Python do projeto, com as dependências do `requirements.txt` da pasta `Predict-do-major`. A pasta de dados pode ser indicada pela variável de ambiente `MAJOR_CS_DADOS`.

## 1. Ambiente e leitura

O CSV mistura séries completas com mapas individuais. Carregamos o identificador do confronto, o placar, o formato, o horário e os jogadores. As estatísticas produzidas durante o jogo ficam fora das entradas do modelo. Os CSVs por tier servem para identificar os grupos de avaliação e reproduzir o histórico da referência.

In [1]:
import hashlib
import json
import os
import sys
from pathlib import Path, PureWindowsPath
from tempfile import TemporaryDirectory
from unittest.mock import patch

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score

candidatas = [Path.cwd(), Path.home() / "Desktop" / "Major" / "Predict-do-major"]
if os.environ.get("MAJOR_CS_DADOS"):
    candidatas.insert(0, Path(os.environ["MAJOR_CS_DADOS"]).expanduser())

PASTA_DADOS = next(
    (pasta.resolve() for pasta in candidatas if (pasta / "cs2_all_tiers_games.csv").is_file()),
    None,
)
if PASTA_DADOS is None:
    raise FileNotFoundError("Indique a pasta Predict-do-major em MAJOR_CS_DADOS.")

# Os módulos originais são usados como dependência local, sem gerar cache na pasta fonte.
pasta_scripts = str(PASTA_DADOS / "scripts")
if pasta_scripts not in sys.path:
    sys.path.insert(0, pasta_scripts)
with patch.object(sys, "dont_write_bytecode", True):
    import reconstruir_major as reconstrucao
    import simular_singapura as simulador
    from modelo_simetrico import ModeloSimetrico

COLUNAS_JOGADORES = [
    f"team{lado}_player{numero}" for lado in (1, 2) for numero in range(1, 6)
]
COLUNAS_DADOS = [
    "match_id", "game_id", "tournament", "datetime", "is_total",
    "team1", "team2", "bestOf", "score1_match", "score2_match", "team1_win",
] + COLUNAS_JOGADORES

caminho_csv = PASTA_DADOS / "cs2_all_tiers_games.csv"
caminho_tier1 = PASTA_DADOS / "cs2_tier1_games.csv"
jogos = pd.read_csv(caminho_csv, usecols=COLUNAS_DADOS, parse_dates=["datetime"])
jogos_tier1 = pd.read_csv(caminho_tier1, usecols=COLUNAS_DADOS, parse_dates=["datetime"])

print(f"Python {sys.version.split()[0]} · pandas {pd.__version__} · scikit-learn {sklearn.__version__}")
display(pd.DataFrame([
    {"base": "Todos os tiers", "linhas": len(jogos),
     "totais de séries": jogos["is_total"].sum(), "mapas": jogos["is_total"].eq(False).sum()},
    {"base": "Tier 1", "linhas": len(jogos_tier1),
     "totais de séries": jogos_tier1["is_total"].sum(), "mapas": jogos_tier1["is_total"].eq(False).sum()},
]))

Python 3.15.2 · pandas 3.1.1 · scikit-learn 1.9.1


,base,linhas,totais de séries,mapas
0,Todos os tiers,24109,11482,12627
1,Tier 1,13044,5194,7850


## 2. Limpeza e alvo

### 2.1 Uma linha por série

`is_total` identifica a linha da série. O `match_id` identifica o confronto; os IDs de equipe do CSV variam de partida para partida, como vimos no notebook original. Mantemos a chave de histórico pelo nome exato da equipe e inspecionamos as repetições de `match_id` antes de acumular os jogos.

In [2]:
totais = jogos.loc[jogos["is_total"].eq(True)].copy()
duplicadas = totais.loc[totais["match_id"].duplicated(keep=False)].copy()
duplicadas["jogadores_preenchidos"] = duplicadas[COLUNAS_JOGADORES].notna().sum(axis=1)
display(duplicadas.sort_values(["match_id", "game_id"])[[
    "match_id", "game_id", "tournament", "team1", "team2", "datetime",
    "score1_match", "score2_match", "team1_win", "jogadores_preenchidos",
]])

,match_id,game_id,tournament,team1,team2,datetime,score1_match,score2_match,team1_win,jogadores_preenchidos
5061,7287899,-153225.0,PGL Astana 2026. Закрытая европейская квалифик...,K27,WW Team,2026-03-31 14:00:00,2.0,0.0,1.0,10
5060,7287899,-153223.0,PGL Astana 2026. Closed European Qualifier,K27,WW Team,2026-03-31 14:00:00,2.0,0.0,0.0,10
5100,7298778,-153207.0,Roman Imperium Cup VII,G2 Esports,BETBOOM TEAM,2026-03-30 20:10:00,1.0,2.0,0.0,10
5099,7298778,-153205.0,Roman Imperium Cup VII,G2 Esports,BetBoom Team,2026-03-30 20:10:00,1.0,2.0,0.0,10
5081,7298808,-153218.0,Roman Imperium Cup VII,Acend,G2 Esports,2026-03-31 00:40:00,0.0,2.0,0.0,10
5080,7298808,-153217.0,Roman Imperium Cup VII,Acend,G2 Esports,2026-03-31 00:40:00,0.0,2.0,0.0,9
4964,7298940,-153267.0,PGL Astana 2026. Закрытая европейская квалифик...,Lavked,100 Thieves,2026-04-02 11:00:00,1.0,2.0,0.0,10
4963,7298940,-153266.0,PGL Astana 2026. Closed European Qualifier,Lavked,100 Thieves,2026-04-02 11:00:00,1.0,2.0,0.0,10


### 2.2 Resolver os quatro duplicados e construir o alvo

- Para `7287899` e `7298940`, mantemos a mesma linha retida pelo notebook original. São repetições da qualificação europeia do PGL Astana.
- Para `7298778`, as escalações coincidem e varia a grafia de BetBoom. Mantemos a grafia `BetBoom Team` nesse registro.
- Para `7298808`, mantemos a linha com a escalação mais completa.

O alvo vale 1 quando `score1_match > score2_match`, e 0 quando o segundo time vence. Retiramos séries sem placar decisivo ou sem identificação completa. Mantemos `bestOf` vazio quando o formato está ausente. A imputação será ajustada só com o treino.

In [3]:
TOTAIS_DESCARTADOS = {
    7287899: -153223,
    7298940: -153266,
    7298778: -153207,
    7298808: -153217,
}
ALVO = "team1_win_corrigido"

def limpar_series(base):
    series = base.loc[base["is_total"].eq(True)].copy()
    duplicatas = series.loc[series["match_id"].duplicated(keep=False)]
    for confronto, par in duplicatas.groupby("match_id"):
        comparaveis = par[["datetime", "bestOf", "score1_match", "score2_match"]].copy()
        comparaveis["team1"] = par["team1"].str.casefold()
        comparaveis["team2"] = par["team2"].str.casefold()
        descarte = par["game_id"].eq(TOTAIS_DESCARTADOS.get(confronto))
        if len(par) != 2 or len(comparaveis.drop_duplicates()) != 1 or descarte.sum() != 1:
            raise ValueError(f"Revise a duplicata da série {confronto}.")
        series = series.drop(index=par.index[descarte])

    assert series["match_id"].is_unique, "Cada série precisa ter um identificador único."
    invalidas = (
        series[["match_id", "datetime", "team1", "team2", "score1_match", "score2_match"]]
        .isna().any(axis=1)
        | series["team1"].eq(series["team2"])
        | series["score1_match"].eq(series["score2_match"])
    )
    rejeitadas = series.loc[invalidas].copy()
    series = series.loc[~invalidas].sort_values(["datetime", "match_id"]).reset_index(drop=True)
    series[ALVO] = series["score1_match"].gt(series["score2_match"]).astype(int)
    return series, rejeitadas

series_v3, rejeitadas = limpar_series(jogos)
series_tier1, rejeitadas_tier1 = limpar_series(jogos_tier1)
assert set(series_tier1["match_id"]).issubset(series_v3["match_id"])
print(f"Séries aproveitadas · todos os tiers {len(series_v3):,} · Tier 1 {len(series_tier1):,}")
print(f"Séries sem formato informado: {series_v3['bestOf'].isna().sum()}")
print(f"Indicadores originais divergentes do placar: {(series_v3['team1_win'].notna() & series_v3['team1_win'].ne(series_v3[ALVO])).sum()}")
display(rejeitadas[["match_id", "team1", "team2", "score1_match", "score2_match"]])

Séries aproveitadas · todos os tiers 11,471 · Tier 1 5,190
Séries sem formato informado: 119
Indicadores originais divergentes do placar: 4891


,match_id,team1,team2,score1_match,score2_match
3117,10095505,США,Канада,NaN,NaN
4771,7299208,FaZe Clan,CYBERSHOKE Esports,1.0,1.0
4830,7298927,NRG Esports,Team Voca,1.0,1.0
4881,7299053,G2 Esports,GamerLegion,1.0,1.0
5067,7298791,Passion UA,FaZe Clan,2.0,2.0
16122,1629909,ALTERNATE aTTaX,THE SUSPECT,1.0,1.0
23528,10064713,NaN,NaN,NaN,NaN


### 2.3 Etiquetar o tier para a avaliação

Cruzamos `match_id` com os três arquivos por tier. A etiqueta se refere à competição registrada no dataset e serve só para diagnóstico. Séries presentes exclusivamente no CSV completo continuam na base com a etiqueta **Sem tier identificado**.

In [4]:
ids_por_tier = {}
for numero in (1, 2, 3):
    dados_tier = pd.read_csv(
        PASTA_DADOS / f"cs2_tier{numero}_games.csv", usecols=["match_id", "is_total"]
    )
    ids_por_tier[f"Tier {numero}"] = set(dados_tier.loc[dados_tier["is_total"], "match_id"])

tier_por_id = {confronto: tier for tier, ids in ids_por_tier.items() for confronto in ids}
assert len(tier_por_id) == sum(map(len, ids_por_tier.values())), "Há IDs em mais de um tier."
series_v3["tier"] = series_v3["match_id"].map(tier_por_id).fillna("Sem tier identificado")
display(series_v3.groupby("tier").agg(
    series=("match_id", "size"), de=("datetime", "min"), ate=("datetime", "max")
))

,series,de,ate
tier,,,
Sem tier identificado,22,2026-11-12 13:00:00,2026-11-14 21:30:00
Tier 1,5190,2023-01-10 09:30:00,2026-11-14 13:30:00
Tier 2,5352,2023-10-05 20:00:00,2026-11-01 22:00:00
Tier 3,907,2023-10-11 16:00:00,2026-11-01 16:00:00


## 3. Histórico disponível antes de cada jogo

### 3.1 Uma perspectiva por equipe

Cada série vira duas participações. Se A venceu, A recebe uma vitória e B recebe uma derrota. Somamos participações e vitórias por nome e horário antes de acumular. Assim, duas partidas com o mesmo horário ficam juntas no histórico e só ficam disponíveis para previsões posteriores.

In [5]:
def acumular_historico(series):
    participacoes = pd.concat([
        series[["datetime", "team1", ALVO]].rename(columns={"team1": "equipe", ALVO: "venceu"}),
        series[["datetime", "team2", ALVO]].rename(columns={"team2": "equipe", ALVO: "venceu"})
        .assign(venceu=lambda tabela: 1 - tabela["venceu"]),
    ], ignore_index=True)
    blocos = (
        participacoes.groupby(["equipe", "datetime"], as_index=False)
        .agg(jogos_no_horario=("venceu", "size"), vitorias_no_horario=("venceu", "sum"))
        .sort_values("datetime")
    )
    por_equipe = blocos.groupby("equipe", sort=False)
    blocos["jogos_anteriores"] = por_equipe["jogos_no_horario"].cumsum()
    blocos["vitorias_anteriores"] = por_equipe["vitorias_no_horario"].cumsum()
    return blocos

historico_v3 = acumular_historico(series_v3)
historico_tier1 = acumular_historico(series_tier1)
assert historico_v3["jogos_no_horario"].sum() == 2 * len(series_v3)
assert historico_v3["vitorias_no_horario"].sum() == len(series_v3)
display(historico_v3.loc[historico_v3["equipe"].eq("100 Thieves")].head(5))

,equipe,datetime,jogos_no_horario,vitorias_no_horario,jogos_anteriores,vitorias_anteriores
19,100 Thieves,2026-02-20 20:10:00,1,0,1,0
20,100 Thieves,2026-02-20 21:30:00,1,1,2,1
21,100 Thieves,2026-02-21 00:30:00,1,1,3,2
22,100 Thieves,2026-02-21 17:30:00,1,0,4,2
23,100 Thieves,2026-03-31 17:00:00,1,0,5,2


### 3.2 Ligar cada confronto ao último bloco estritamente anterior

Para cada lado, buscamos o último acumulado com horário menor que o do jogo. Uma estreia começa com zero jogos, zero vitórias e taxa ausente. A referência consulta só o histórico Tier 1, inclusive quando é usada para prever uma partida de Tier 2 ou 3. A V3 consulta o histórico completo para os mesmos confrontos.

In [6]:
CONTAGENS = ["jogos_anteriores", "vitorias_anteriores"]
ATRIBUTOS_HISTORICO = CONTAGENS + ["taxa_vitorias_anteriores"]

def juntar_historico(series, historico):
    saida = series.copy()
    consulta = historico[["equipe", "datetime"] + CONTAGENS].rename(
        columns={"datetime": "horario_historico"}
    )
    for lado in (1, 2):
        time = f"team{lado}"
        atual = series[["match_id", "datetime", time]].rename(columns={time: "equipe"})
        anterior = pd.merge_asof(
            atual.sort_values("datetime"), consulta,
            left_on="datetime", right_on="horario_historico", by="equipe",
            direction="backward", allow_exact_matches=False,
        )
        tem_historico = anterior["horario_historico"].notna()
        assert anterior.loc[tem_historico, "horario_historico"].lt(
            anterior.loc[tem_historico, "datetime"]
        ).all()
        anterior[CONTAGENS] = anterior[CONTAGENS].fillna(0).astype(int)
        anterior["taxa_vitorias_anteriores"] = (
            anterior["vitorias_anteriores"] / anterior["jogos_anteriores"].replace(0, np.nan)
        )
        colunas = {atributo: f"{time}_{atributo}" for atributo in ATRIBUTOS_HISTORICO}
        saida = saida.merge(
            anterior[["match_id"] + ATRIBUTOS_HISTORICO].rename(columns=colunas),
            on="match_id", validate="one_to_one",
        )
    for atributo in ATRIBUTOS_HISTORICO:
        saida[f"dif_{atributo}"] = saida[f"team1_{atributo}"] - saida[f"team2_{atributo}"]
    return saida

dados_v3 = juntar_historico(series_v3, historico_v3)
dados_ref = juntar_historico(series_v3, historico_tier1)
assert (dados_v3["team1_jogos_anteriores"] >= dados_ref["team1_jogos_anteriores"]).all()
assert (dados_v3["team2_jogos_anteriores"] >= dados_ref["team2_jogos_anteriores"]).all()
display(dados_v3[["datetime", "team1", "team2", *reconstrucao.V1_COLUMNS, ALVO]].tail(5))

,datetime,team1,team2,dif_jogos_anteriores,dif_vitorias_anteriores,dif_taxa_vitorias_anteriores,bestOf,team1_win_corrigido
11466,2026-11-14 16:15:00,Team Buster,Team WorldEdit,27,16,-0.025000,3.0,1
11467,2026-11-14 16:15:00,Team leva2k,Team NS,-1,0,0.500000,3.0,1
11468,2026-11-14 18:00:00,Iberian Soul,FORZE Reload,-1,5,0.087814,3.0,1
11469,2026-11-14 19:00:00,Team PCH3LK1N,Team Jerry,2,-2,-0.317460,3.0,1
11470,2026-11-14 21:30:00,WW Team,Hype Tech,63,33,NaN,3.0,0


## 4. Ranking HLTV e parâmetros da referência

### 4.1 Pontos conhecidos antes da partida

Reusamos a regra do projeto. A edição precisa ter sido publicada antes do dia do confronto, ter no máximo 14 dias e combinar nome com pelo menos três jogadores em comum. A diferença de pontos fica ausente se um dos lados não puder ser confirmado. A validação histórica usa a escalação registrada no CSV. A simulação usa a escalação publicada para o Major. As equivalências intermediárias são produzidas em pasta temporária.

In [7]:
with TemporaryDirectory(prefix="major-v3-ranking-") as pasta_temporaria:
    with (
        patch.object(reconstrucao, "PROJECT", PASTA_DADOS),
        patch.object(reconstrucao, "OUTPUT", Path(pasta_temporaria)),
    ):
        atributos_ranking, auditoria_ranking = reconstrucao.ranking_features(series_v3)

assert len(atributos_ranking) == len(series_v3)
pontuacao = atributos_ranking[["match_id", "dif_pontos_hltv"]]
dados_v3 = dados_v3.merge(pontuacao, on="match_id", validate="one_to_one")
dados_ref = dados_ref.merge(pontuacao, on="match_id", validate="one_to_one")

display(dados_v3.groupby("tier")["dif_pontos_hltv"].agg(
    series="size", ranking_dois_lados="count", cobertura=lambda pontos: pontos.notna().mean()
).round(3))

,series,ranking_dois_lados,cobertura
tier,,,
Sem tier identificado,22,1,0.045
Tier 1,5190,3446,0.664
Tier 2,5352,1609,0.301
Tier 3,907,141,0.155


### 4.2 Conferir a reprodução da versão anterior

O manifesto salvo informa quais atributos foram usados. Conferimos cada um deles, mais o alvo, nas séries Tier 1 do experimento anterior. Essa verificação testa se a V3 poderá ser comparada à referência sem mudar a definição das entradas.

In [8]:
relatorio_ref = json.loads(
    (PASTA_DADOS / "dados_externos" / "reconstrucao" / "ultimo_experimento.json").read_bytes()
)
pasta_experimento = PASTA_DADOS / "experimentos" / PureWindowsPath(
    relatorio_ref["pasta_experimento"]
).name
selecao_ref = json.loads((pasta_experimento / "selecao_antes_do_teste.json").read_bytes())
ENTRADAS = selecao_ref["entradas"]
assert selecao_ref["modelo"] == "V1 + ranking (simetrico)"
assert ENTRADAS == reconstrucao.V1_COLUMNS + ["dif_pontos_hltv"]

colunas_comparadas = ENTRADAS + [ALVO]
base_ref_salva = pd.read_csv(
    pasta_experimento / "base_experimentos.csv", usecols=["match_id", *colunas_comparadas]
).set_index("match_id").sort_index()[colunas_comparadas]
base_ref_recalculada = dados_ref.loc[dados_ref["tier"].eq("Tier 1")].set_index("match_id")
pd.testing.assert_frame_equal(
    base_ref_recalculada.sort_index()[colunas_comparadas], base_ref_salva,
    check_dtype=False, check_exact=False, rtol=1e-10, atol=1e-12,
)
modelo_ref = joblib.load(pasta_experimento / "modelo_selecionado.joblib")
assert list(modelo_ref.modelo.feature_names_in_) == ENTRADAS
print(f"{len(base_ref_salva):,} séries Tier 1 conferidas atributo por atributo.")
print(ENTRADAS)

5,190 séries Tier 1 conferidas atributo por atributo.
['dif_jogos_anteriores', 'dif_vitorias_anteriores', 'dif_taxa_vitorias_anteriores', 'bestOf', 'dif_pontos_hltv']


## 5. Separação cronológica e treino

### 5.1 Manter os cortes do primeiro notebook

| Conjunto | Período | Uso |
|---|---|---|
| Treino | Antes de 01/01/2026 | Ajustar os parâmetros e as transformações |
| Validação | De 01/01/2026 até 30/06/2026 | Comparar o comportamento das versões |
| Teste | A partir de 01/07/2026 | Medir o resultado fora do período de treino |

A avaliação usa o horário registrado no CSV como referência temporal. Uma série do teste pode consultar resultados de séries com horário anterior, inclusive no mesmo período. O modelo e o preenchimento de valores ausentes ficam congelados durante a avaliação.

In [9]:
INICIO_VALIDACAO = pd.Timestamp("2026-01-01")
INICIO_TESTE = pd.Timestamp("2026-07-01")
data = dados_v3["datetime"]
particoes = {
    "Treino": data.lt(INICIO_VALIDACAO),
    "Validação": data.ge(INICIO_VALIDACAO) & data.lt(INICIO_TESTE),
    "Teste": data.ge(INICIO_TESTE),
}
assert dados_v3["match_id"].equals(dados_ref["match_id"])
assert (sum(mascara.astype(int) for mascara in particoes.values()) == 1).all()
display(pd.DataFrame([
    {"conjunto": nome, "total": int(mascara.sum()),
     "Tier 1": int((mascara & dados_v3["tier"].eq("Tier 1")).sum()),
     "Tier 2 e 3": int((mascara & dados_v3["tier"].isin(["Tier 2", "Tier 3"])).sum()),
     "Sem tier identificado": int((mascara & dados_v3["tier"].eq("Sem tier identificado")).sum()),
     "de": data[mascara].min(), "até": data[mascara].max()}
    for nome, mascara in particoes.items()
]).set_index("conjunto"))

,total,Tier 1,Tier 2 e 3,Sem tier identificado,de,até
conjunto,,,,,,
Treino,8109,3775,4334,0,2023-01-10 09:30:00,2025-12-28 18:15:00
Validação,1844,642,1202,0,2026-01-03 08:00:00,2026-06-30 21:20:00
Teste,1518,773,723,22,2026-07-01 11:45:00,2026-11-14 21:30:00


### 5.2 Treinar a V3 com o mesmo pipeline

O pipeline preenche valores ausentes com a mediana e acrescenta indicadores de ausência, padroniza as entradas e treina a regressão logística com `max_iter=1000`. Usamos a mesma função de criação e conferimos os parâmetros contra o modelo salvo. A probabilidade final usa as duas ordens da partida, com `bestOf` fixo e diferenças invertidas.

In [10]:
treino = dados_v3.loc[particoes["Treino"]]
pipeline_v3 = reconstrucao.new_pipeline()
for nome, etapa in pipeline_v3.named_steps.items():
    parametros_ref = modelo_ref.modelo.named_steps[nome].get_params(deep=False)
    assert json.dumps(etapa.get_params(deep=False), sort_keys=True) == json.dumps(parametros_ref, sort_keys=True)

pipeline_v3.fit(treino[ENTRADAS], treino[ALVO])
modelo_v3 = ModeloSimetrico(pipeline_v3)

amostra = treino[ENTRADAS].head(30)
invertida = amostra.copy()
diferenças = [nome for nome in ENTRADAS if nome.startswith("dif_")]
invertida[diferenças] = -invertida[diferenças]
np.testing.assert_allclose(
    modelo_v3.predict_proba(amostra)[:, 1] + modelo_v3.predict_proba(invertida)[:, 1], 1.0,
)
print(f"V3 treinada com {len(treino):,} séries.")
display(pd.DataFrame({
    "atributo": pipeline_v3.named_steps["imputacao"].statistics_,
}, index=ENTRADAS).rename(columns={"atributo": "mediana no treino"}))

V3 treinada com 8,109 séries.


,mediana no treino
dif_jogos_anteriores,3.000000
dif_vitorias_anteriores,2.000000
dif_taxa_vitorias_anteriores,0.015385
bestOf,3.000000
dif_pontos_hltv,11.000000


## 6. Avaliar se os times com menos histórico ganham cobertura

### 6.1 Definir os recortes e medir probabilidades

Usamos **menos de 10 séries anteriores de Tier 1** como critério operacional de pouco histórico. A mesma regra vale para as duas versões, em cada data. Também comparamos as séries de Tier 2 e 3, que podem incluir equipes bem conhecidas do Tier 1.

Log Loss e Brier medem a qualidade da probabilidade e devem diminuir. Acurácia mede quantos vencedores foram apontados corretamente e AUC mede a ordenação dos favoritos. Incluímos 50% para cada lado como referência simples. A comparação da referência com a V3 sempre usa as mesmas séries.

In [11]:
LIMITE_POUCO_HISTORICO = 10
GRUPO_POUCO_HISTORICO = "Pelo menos um lado com menos de 10 séries Tier 1"
colunas_volume = [f"team{lado}_jogos_anteriores" for lado in (1, 2)]
pouco_historico = dados_ref[colunas_volume].lt(LIMITE_POUCO_HISTORICO)
grupos = {
    "Todas as séries": pd.Series(True, index=dados_v3.index),
    "Apenas Tier 1": dados_v3["tier"].eq("Tier 1"),
    "Tier 2 e 3": dados_v3["tier"].isin(["Tier 2", "Tier 3"]),
    GRUPO_POUCO_HISTORICO: pouco_historico.any(axis=1),
}

def calcular_metricas(alvo, probabilidade):
    return {
        "Log Loss": log_loss(alvo, probabilidade, labels=[0, 1]),
        "Brier": brier_score_loss(alvo, probabilidade),
        "Acurácia": accuracy_score(alvo, (probabilidade > 0.5).astype(int)),
        "AUC": roc_auc_score(alvo, probabilidade) if alvo.nunique() == 2 else np.nan,
    }

def avaliar(conjunto):
    selecao = particoes[conjunto]
    alvo = dados_v3.loc[selecao, ALVO]
    probs = pd.DataFrame({
        "Referência HLTV": modelo_ref.predict_proba(dados_ref.loc[selecao, ENTRADAS])[:, 1],
        "V3": modelo_v3.predict_proba(dados_v3.loc[selecao, ENTRADAS])[:, 1],
        "50%": 0.5,
    }, index=alvo.index)
    resultados = []
    for nome, pertence in grupos.items():
        ids = alvo.index[pertence.loc[alvo.index]]
        if len(ids) == 0:
            continue
        for versao in probs.columns:
            resultados.append({
                "grupo": nome, "modelo": versao, "séries": len(ids),
                **calcular_metricas(alvo.loc[ids], probs.loc[ids, versao]),
            })
    return pd.DataFrame(resultados).set_index(["grupo", "modelo"]), probs

metricas_validacao, prob_validacao = avaliar("Validação")
display(metricas_validacao.round(4))

séries  ...     AUC
grupo                                            modelo                   ...        
Todas as séries                                  Referência HLTV    1844  ...  0.6100
                                                 V3                 1844  ...  0.6327
                                                 50%                1844  ...  0.5000
Apenas Tier 1                                    Referência HLTV     642  ...  0.6433
                                                 V3                  642  ...  0.6379
                                                 50%                 642  ...  0.5000
Tier 2 e 3                                       Referência HLTV    1202  ...  0.5926
                                                 V3                 1202  ...  0.6337
                                                 50%                1202  ...  0.5000
Pelo menos um lado com menos de 10 séries Tier 1 Referência HLTV    1121  ...  0.6103
                                                 V3                 1121  ...  0.6315
                                                 50%                1121  ...  0.5000

[12 rows x 5 columns]

### 6.2 Ler o teste congelado

Para verificar a comparação, o Log Loss da referência no Tier 1 deve reproduzir o teste arquivado. O recorte Tier 1 usa as mesmas séries do teste reportado no notebook original. O quadro inclui também os demais tiers disponíveis no CSV completo.

In [12]:
metricas_teste, prob_teste = avaliar("Teste")
metrica_ref_arquivada = next(
    linha for linha in relatorio_ref["metricas_teste_final"]
    if linha["modelo"] == selecao_ref["modelo"]
)
np.testing.assert_allclose(
    metricas_teste.loc[("Apenas Tier 1", "Referência HLTV"), "Log Loss"],
    metrica_ref_arquivada["log_loss"],
    atol=1e-9, rtol=0,
)
display(metricas_teste.round(4))

séries  ...     AUC
grupo                                            modelo                   ...        
Todas as séries                                  Referência HLTV    1518  ...  0.6389
                                                 V3                 1518  ...  0.6460
                                                 50%                1518  ...  0.5000
Apenas Tier 1                                    Referência HLTV     773  ...  0.6350
                                                 V3                  773  ...  0.6520
                                                 50%                 773  ...  0.5000
Tier 2 e 3                                       Referência HLTV     723  ...  0.6492
                                                 V3                  723  ...  0.6473
                                                 50%                 723  ...  0.5000
Pelo menos um lado com menos de 10 séries Tier 1 Referência HLTV     864  ...  0.6607
                                                 V3                  864  ...  0.6620
                                                 50%                 864  ...  0.5000

[12 rows x 5 columns]

### 6.3 Dar o mesmo peso a cada equipe com pouco histórico

Ao avaliar as participações das equipes com menos de 10 séries de Tier 1, calculamos o Brier por equipe e tiramos a média entre equipes. Isso reduz o peso de quem aparece em muitos jogos. A cobertura mostra a proporção de participações com pelo menos uma partida anterior disponível. O desvio médio é a probabilidade prevista menos a frequência observada de vitórias, em pontos percentuais.

In [13]:
participacoes_teste = []
for lado in (1, 2):
    coluna_equipe = f"team{lado}"
    coluna_jogos = f"{coluna_equipe}_jogos_anteriores"
    ids = dados_ref.index[particoes["Teste"] & pouco_historico[coluna_jogos]]
    tabela = pd.DataFrame({
        "equipe": dados_v3.loc[ids, coluna_equipe],
        "histórico referência": dados_ref.loc[ids, coluna_jogos],
        "histórico V3": dados_v3.loc[ids, coluna_jogos],
        "venceu": dados_v3.loc[ids, ALVO],
        "prob referência": prob_teste.loc[ids, "Referência HLTV"],
        "prob V3": prob_teste.loc[ids, "V3"],
    })
    if lado == 2:
        colunas_invertidas = ["venceu", "prob referência", "prob V3"]
        tabela[colunas_invertidas] = 1.0 - tabela[colunas_invertidas]
    participacoes_teste.append(tabela)

participacoes_teste = pd.concat(participacoes_teste, ignore_index=True)
diagnostico_equipes = []
for versao in ("referência", "V3"):
    erro = participacoes_teste[f"prob {versao}"] - participacoes_teste["venceu"]
    por_equipe = participacoes_teste.assign(erro=erro, quadratico=erro.pow(2)).groupby("equipe")
    diagnostico_equipes.append({
        "modelo": versao,
        "equipes": por_equipe.ngroups,
        "participações": len(participacoes_teste),
        "cobertura de histórico (%)": 100 * participacoes_teste[f"histórico {versao}"].gt(0).mean(),
        "Brier médio por equipe": por_equipe["quadratico"].mean().mean(),
        "desvio médio por equipe (p.p.)": 100 * por_equipe["erro"].mean().mean(),
    })
diagnostico_equipes = pd.DataFrame(diagnostico_equipes).set_index("modelo")
display(diagnostico_equipes.round(4))

,equipes,participações,cobertura de histórico (%),Brier médio por equipe,desvio médio por equipe (p.p.)
modelo,,,,,
referência,309,1335,46.2921,0.2361,11.0395
V3,309,1335,85.3933,0.2303,10.1442


## 7. Salvar e verificar o modelo

Guardamos o pipeline ajustado e a lista ordenada de entradas. A simetrização é aplicada pelo mesmo adaptador do projeto ao carregar o arquivo. O registro inclui a versão do scikit-learn, o hash do CSV e os cortes da avaliação. O arquivo `modelo_v3.joblib` é gravado no diretório de execução, configurável por `MAJOR_CS_SAIDA`.

In [14]:
pasta_saida = Path(os.environ.get("MAJOR_CS_SAIDA", Path.cwd())).resolve()
pasta_saida.mkdir(parents=True, exist_ok=True)
arquivo_modelo = pasta_saida / "modelo_v3.joblib"
pacote = {
    "versao": "V3",
    "pipeline": pipeline_v3,
    "entradas": ENTRADAS,
    "simetrizacao": "media_da_previsao_direta_e_complemento_da_inversa",
    "treino_antes_de": str(INICIO_VALIDACAO.date()),
    "teste_a_partir_de": str(INICIO_TESTE.date()),
    "sklearn": sklearn.__version__,
    "csv_sha256": hashlib.sha256(caminho_csv.read_bytes()).hexdigest(),
}
with TemporaryDirectory(prefix="major-v3-modelo-", dir=pasta_saida) as temporaria:
    caminho_temporario = Path(temporaria) / arquivo_modelo.name
    joblib.dump(pacote, caminho_temporario, compress=3)
    recuperado = joblib.load(caminho_temporario)
    verificado = ModeloSimetrico(recuperado["pipeline"])
    np.testing.assert_allclose(
        verificado.predict_proba(amostra[recuperado["entradas"]]), modelo_v3.predict_proba(amostra)
    )
    caminho_temporario.replace(arquivo_modelo)
print(f"Modelo salvo e recarregado em {arquivo_modelo}")

Modelo salvo e recarregado em C:\Users\franc\Documents\ChatGPT\Major cs\modelo_v3.joblib


## 8. Aplicar ao Major de Singapura

### 8.1 Congelar os dados na data da previsão

Usamos a configuração oficial arquivada e o ranking de 16/11/2026. O simulador do projeto confirma a identidade do time por nome e escalação recente antes de consultar o histórico. Rodamos essa preparação com as séries de todos os tiers. Para quantificar a mudança de cobertura, repetimos a preparação com as séries Tier 1.

In [15]:
pasta_major = PASTA_DADOS / "singapura_2026"
ponteiro = json.loads((pasta_major / "ultima_preparacao.json").read_bytes())
fontes_major = pasta_major / PureWindowsPath(ponteiro["pasta_fontes"]).name
configuracao = json.loads((fontes_major / "config_torneio.json").read_bytes())

with patch.object(simulador, "PROJECT", PASTA_DADOS):
    with patch.object(simulador, "base_v1", return_value=(jogos, series_v3)):
        atributos_major, cobertura_major, _ = simulador.snapshots(configuracao, fontes_major)
    with patch.object(simulador, "base_v1", return_value=(jogos_tier1, series_tier1)):
        _, cobertura_major_ref, _ = simulador.snapshots(configuracao, fontes_major)

cobertura_fase1 = (
    cobertura_major.loc[cobertura_major["etapa_entrada"].eq(1)]
    .merge(
        cobertura_major_ref[["chave", "jogos_historico_local", "nome_local"]],
        on="chave", validate="one_to_one", suffixes=("_v3", "_ref"),
    )
    .sort_values("seed_entrada")
)
cobertura_antes = int(cobertura_fase1["nome_local_ref"].notna().sum())
cobertura_depois = int(cobertura_fase1["nome_local_v3"].notna().sum())
print(f"Histórico corroborado na Etapa 1 · referência {cobertura_antes}/16 · V3 {cobertura_depois}/16")
display(cobertura_fase1[[
    "nome_oficial", "nome_local_v3", "jogos_historico_local_ref",
    "jogos_historico_local_v3", "ranking_pontos",
]])

Histórico corroborado na Etapa 1 · referência 8/16 · V3 10/16


,nome_oficial,nome_local_v3,jogos_historico_local_ref,jogos_historico_local_v3,ranking_pontos
0,Ninjas in Pyjamas,Ninjas in Pyjamas,140,229,93
1,PVISION,PARIVISION,144,269,188
2,HEROIC,Heroic,223,274,76
3,FaZe Clan,FaZe Clan,242,268,90
4,Team Liquid,Team Liquid,203,229,128
5,Eternal Fire,Eternal Fire,113,216,76
6,100 Thieves,100 Thieves,0,47,55
7,9z Globant,9z Team,101,123,89
8,M80,M80,119,151,71
9,DENDELE,NaN,0,0,48


### 8.2 Rodar 100.000 simulações com a V3

Rodamos **100.000 torneios simulados** com a **semente 25**. O simulador usa as probabilidades da V3, o chaveamento suíço e o formato de cada série. A chance de passar mede quantas vezes cada equipe chega à Etapa 2. Os três gráficos reutilizam as campanhas registradas nessas mesmas simulações.

A coluna **Referência (%)** vem da previsão arquivada com **20.000 simulações**.

In [16]:
resultado_ref = json.loads((pasta_major / "ultima_simulacao.json").read_bytes())
assert resultado_ref["modelo"] == selecao_ref["modelo"]
assert resultado_ref["data_previsao"] == configuracao["data_previsao"]
repeticoes = 100_000
semente = resultado_ref["semente_aleatoria"]

chaves, matrizes, _ = simulador.probability_matrices(
    configuracao, atributos_major, verificado, ENTRADAS
)
# Registramos a campanha de cada equipe enquanto o simulador roda.
campanhas = [(3, 0), (3, 1), (3, 2), (2, 3), (1, 3), (0, 3)]
indice_campanha = {campanha: i for i, campanha in enumerate(campanhas)}
chaves_etapa1 = configuracao["etapas"]["1"]
indice_equipe = {chave: i for i, chave in enumerate(chaves)}
equipe_etapa1 = [indice_equipe[chave] for chave in chaves_etapa1]
perfis_etapa1 = []
suico_original = simulador.swiss_stage


def suico_com_registro(equipes, etapa, matrizes, rng, prioridade, registrar):
    if etapa != 1:
        return suico_original(equipes, etapa, matrizes, rng, prioridade, registrar)

    series_jogadas = dict.fromkeys(equipes, 0)

    def registrar_partida(time_a, time_b, etapa_atual):
        series_jogadas[time_a] += 1
        series_jogadas[time_b] += 1
        registrar(time_a, time_b, etapa_atual)

    classificados = suico_original(
        equipes, etapa, matrizes, rng, prioridade, registrar_partida,
    )
    avancaram = set(classificados)
    perfil = []
    for equipe in equipe_etapa1:
        total = series_jogadas[equipe]
        vitorias = 3 if equipe in avancaram else total - 3
        perfil.append(indice_campanha[vitorias, total - vitorias])
    perfis_etapa1.append(tuple(perfil))
    return classificados


with patch.object(simulador, "swiss_stage", suico_com_registro):
    contagens, _, _ = simulador.monte_carlo(
        configuracao, chaves, matrizes, iterations=repeticoes, seed=semente,
    )

cenarios_etapa1 = np.asarray(perfis_etapa1, dtype=np.int8)
assert cenarios_etapa1.shape == (repeticoes, 16)
for i, vagas in enumerate((2, 3, 3, 3, 3, 2)):
    assert np.all((cenarios_etapa1 == i).sum(axis=1) == vagas)
np.testing.assert_array_equal(
    (cenarios_etapa1 < 3).sum(axis=0), contagens[equipe_etapa1, 0],
)
fases = [
    "avançar à Etapa 2", "avançar à Etapa 3", "quartas",
    "semifinal", "final", "título", "terceiro lugar",
]
chances = pd.DataFrame(100 * contagens / repeticoes, index=chaves, columns=fases)
chances = chances.rename_axis("chave").reset_index().merge(
    cobertura_major[["chave", "nome_oficial", "etapa_entrada"]],
    on="chave", validate="one_to_one",
)
pasta_simulacao_ref = pasta_major / PureWindowsPath(resultado_ref["pasta_simulacao"]).name
chances_ref = pd.read_csv(
    pasta_simulacao_ref / "chances_por_equipe.csv",
    usecols=["chave", "alcancar_etapa2_ou_superior"],
)
primeira_fase = (
    chances.loc[chances["etapa_entrada"].eq(1)]
    .merge(chances_ref, on="chave", validate="one_to_one")
    .rename(columns={"avançar à Etapa 2": "V3 (%)"})
    .assign(**{"Referência (%)": lambda tabela: 100 * tabela["alcancar_etapa2_ou_superior"]})
    .sort_values("V3 (%)", ascending=False)
    .reset_index(drop=True)
)
assert len(primeira_fase) == 16
assert np.isclose(primeira_fase["V3 (%)"].sum(), 800.0)
primeira_fase["Diferença (p.p.)"] = primeira_fase["V3 (%)"] - primeira_fase["Referência (%)"]
display(primeira_fase[["nome_oficial", "V3 (%)", "Referência (%)", "Diferença (p.p.)"]].round(3))

Simulacoes: 5000/100000
Simulacoes: 10000/100000
Simulacoes: 15000/100000
Simulacoes: 20000/100000
Simulacoes: 25000/100000
Simulacoes: 30000/100000
Simulacoes: 35000/100000
Simulacoes: 40000/100000
Simulacoes: 45000/100000
Simulacoes: 50000/100000
Simulacoes: 55000/100000
Simulacoes: 60000/100000
Simulacoes: 65000/100000
Simulacoes: 70000/100000
Simulacoes: 75000/100000
Simulacoes: 80000/100000
Simulacoes: 85000/100000
Simulacoes: 90000/100000
Simulacoes: 95000/100000
Simulacoes: 100000/100000


,nome_oficial,V3 (%),Referência (%),Diferença (p.p.)
0,PVISION,68.298,59.200,9.098
1,FaZe Clan,62.321,65.910,-3.589
2,Team Liquid,62.095,62.945,-0.850
3,HEROIC,60.683,60.960,-0.277
4,Ninjas in Pyjamas,57.474,49.185,8.289
5,Eternal Fire,55.347,46.235,9.112
6,DENDELE,46.658,47.265,-0.607
7,Wildcard,46.069,46.360,-0.291
8,Imperial Esports,46.035,46.430,-0.395
9,Lynn Vision Gaming,45.711,45.785,-0.074


## 9. Interpretar a V3

Mais partidas significam mais observações para equipes pouco representadas. A taxa acumulada continua dando o mesmo peso a uma vitória em qualquer tier e pode misturar adversários de forças diferentes. Os recortes de teste ajudam a verificar o efeito disso. A base histórica local vai até 14/11/2026 e o ranking da simulação é de 16/11; os percentuais descrevem o cenário pré-evento com essas fontes.

In [17]:
ref_pequenos = metricas_teste.loc[(GRUPO_POUCO_HISTORICO, "Referência HLTV"), "Log Loss"]
v3_pequenos = metricas_teste.loc[(GRUPO_POUCO_HISTORICO, "V3"), "Log Loss"]
ref_tier1 = metricas_teste.loc[("Apenas Tier 1", "Referência HLTV"), "Log Loss"]
v3_tier1 = metricas_teste.loc[("Apenas Tier 1", "V3"), "Log Loss"]

print(f"Log Loss no teste com pouco histórico · referência {ref_pequenos:.4f} · V3 {v3_pequenos:.4f}")
print(f"Log Loss no teste Tier 1 · referência {ref_tier1:.4f} · V3 {v3_tier1:.4f}")
print(f"Cobertura na Etapa 1 · referência {cobertura_antes}/16 · V3 {cobertura_depois}/16")
print("Oito maiores chances da V3 · " + ", ".join(primeira_fase["nome_oficial"].head(8)))

Log Loss no teste com pouco histórico · referência 0.6609 · V3 0.6472
Log Loss no teste Tier 1 · referência 0.6864 · V3 0.6549
Cobertura na Etapa 1 · referência 8/16 · V3 10/16
Oito maiores chances da V3 · PVISION, FaZe Clan, Team Liquid, HEROIC, Ninjas in Pyjamas, Eternal Fire, DENDELE, Wildcard


In [18]:
# Gráfico 1 — chances individuais por campanha na Etapa 1
from html import escape

from IPython.display import HTML

ordem_pelo_resumo = {chave: i for i, chave in enumerate(chaves_etapa1)}
ordem = primeira_fase["chave"].map(ordem_pelo_resumo).to_numpy()
cenarios = cenarios_etapa1[:, ordem]
nomes = primeira_fase["nome_oficial"].tolist()
placares = [f"{vitorias}–{derrotas}" for vitorias, derrotas in campanhas]
contagem_campanhas = np.column_stack(
    [(cenarios == codigo).sum(axis=0) for codigo in range(len(campanhas))]
)
avancos = contagem_campanhas[:, :3].sum(axis=1)
np.testing.assert_array_equal(contagem_campanhas.sum(axis=1), np.full(len(nomes), repeticoes))
np.testing.assert_array_equal(100 * avancos / repeticoes, primeira_fase["V3 (%)"].to_numpy())


def inteiro_pt(numero):
    return f"{numero:,}".replace(",", ".")


def percentual(ocorrencias):
    return f"{100 * ocorrencias / repeticoes:.3f}%".replace(".", ",")


amostra = f"V3 · {inteiro_pt(repeticoes)} simulações · semente {semente}"
estilo_painel = "background:#fff;color:#172a34;padding:8px 4px;font:14px system-ui,sans-serif"
estilo_tabela = (
    "background:#fff;color:#172a34;border-collapse:separate;border-spacing:3px;"
    "font:13px system-ui,sans-serif;font-variant-numeric:tabular-nums;text-align:center"
)
verde = "99,188,151"
roxo = "170,147,215"

cabecalho = "".join(
    f'<th scope="col" style="padding:6px 10px">{placar}</th>' for placar in placares
)
linhas = []
maximos = contagem_campanhas.max(axis=0)
teto = maximos.max()
destaques = []
for coluna in (0, 5):
    lideres = ", ".join(
        escape(nome) for nome, vezes in zip(nomes, contagem_campanhas[:, coluna], strict=True)
        if vezes == maximos[coluna]
    )
    destaques.append(f'<strong>{placares[coluna]}</strong> · {lideres} ({percentual(maximos[coluna])})')

for nome, valores, avanco in zip(nomes, contagem_campanhas, avancos, strict=True):
    celulas = []
    for coluna, ocorrencias in enumerate(valores):
        rgb = verde if coluna < 3 else roxo
        intensidade = 0.12 + 0.88 * ocorrencias / teto
        estilo = f"background:rgba({rgb},{intensidade:.3f});padding:5px 9px;border-radius:4px"
        if coluna in (0, 5) and ocorrencias == maximos[coluna]:
            estilo += ";font-weight:750;outline:2px solid #374c58;outline-offset:-2px"
        celulas.append(f'<td style="{estilo}">{percentual(ocorrencias)}</td>')
    celulas.append(
        f'<td style="background:#e0f2ea;padding:5px 9px;font-weight:700">{percentual(avanco)}</td>'
    )
    linhas.append(
        '<tr><th scope="row" style="text-align:left;padding:5px 10px;white-space:nowrap">'
        f'{escape(nome)}</th>{"".join(celulas)}</tr>'
    )

display(HTML(f"""
<div style="{estilo_painel}">
<h3 style="margin:0 0 4px">1. Chances por time</h3>
<p style="margin:0 0 10px">{amostra}</p>
<p style="margin:0 0 8px">{" &nbsp;|&nbsp; ".join(destaques)}</p>
<div style="overflow-x:auto">
<table style="{estilo_tabela};min-width:720px">
    <thead>
        <tr>
            <th rowspan="2" scope="col" style="text-align:left;padding:6px 10px">Equipe</th>
            <th colspan="3" scope="colgroup" style="background:#dcefe7;padding:6px 10px">Classificação</th>
            <th colspan="3" scope="colgroup" style="background:#ece4f5;padding:6px 10px">Eliminação</th>
            <th rowspan="2" scope="col" style="background:#dcefe7;padding:6px 10px">Avança (V3)</th>
        </tr>
        <tr>{cabecalho}</tr>
    </thead>
    <tbody>{"".join(linhas)}</tbody>
</table>
</div>
</div>
"""))

In [19]:
# Gráfico 2 — duplas 3–0 e 0–3 no mesmo cenário
from collections import Counter

duplas_3_0 = [tuple(linha) for linha in np.where(cenarios == 0)[1].reshape(-1, 2)]
duplas_0_3 = [tuple(linha) for linha in np.where(cenarios == 5)[1].reshape(-1, 2)]
contagem_3_0 = Counter(duplas_3_0)
contagem_0_3 = Counter(duplas_0_3)
contagem_extremos = Counter(zip(duplas_3_0, duplas_0_3, strict=True))
assert all(sum(totais.values()) == repeticoes for totais in (contagem_3_0, contagem_0_3, contagem_extremos))


def nome_da_dupla(dupla):
    return " + ".join(nomes[i] for i in dupla)


def mapa_de_duplas(contagem, placar, cor):
    maior = max(contagem.values())
    lideres = [nome_da_dupla(dupla) for dupla, vezes in contagem.items() if vezes == maior]
    lideres_html = " / ".join(escape(nome) for nome in lideres)
    titulo = "Dupla mais frequente" if len(lideres) == 1 else "Duplas empatadas na maior frequência"
    cabecalho = "".join(
        f'<th scope="col" style="min-width:58px;max-width:80px;padding:6px 3px;'
        f'font-size:11px;line-height:1.2">{escape(nome)}</th>'
        for nome in nomes
    )
    linhas = []
    for i, nome in enumerate(nomes):
        linha = [
            f'<th scope="row" style="text-align:left;white-space:nowrap;padding:4px 8px">'
            f'{escape(nome)}</th>'
        ]
        for j in range(len(nomes)):
            if j <= i:
                linha.append('<td style="background:#f5f7f7"></td>')
                continue
            dupla = (i, j)
            vezes = contagem[dupla]
            intensidade = 0.08 + 0.82 * vezes / maior
            estilo = (
                f"background-color:rgba({cor},{intensidade:.3f});"
                "font-size:11px;padding:5px 3px;white-space:nowrap"
            )
            if vezes == maior:
                estilo += ";font-weight:750;outline:2px solid #374c58;outline-offset:-2px"
            dica = escape(f"{nome_da_dupla(dupla)} · {inteiro_pt(vezes)} simulações")
            linha.append(f'<td style="{estilo}" title="{dica}">{percentual(vezes)}</td>')
        linhas.append(f'<tr>{"".join(linha)}</tr>')

    return f"""
    <section style="margin:14px 0">
        <h4 style="margin:0 0 4px">Ambos {placar}</h4>
        <p style="margin:4px 0 8px">{titulo} · <strong>{lideres_html}</strong>
            · <strong>{percentual(maior)}</strong> ({inteiro_pt(maior)} simulações)</p>
        <div style="overflow-x:auto">
        <table style="{estilo_tabela}" aria-label="Chance dos dois times terminarem {placar}">
            <thead><tr><th scope="col" style="text-align:left;padding:6px 8px">Equipe</th>{cabecalho}</tr></thead>
            <tbody>{"".join(linhas)}</tbody>
        </table>
        </div>
    </section>
    """


linhas_extremos = []
for (invictos, sem_vitorias), vezes in contagem_extremos.most_common(5):
    linhas_extremos.append(
        f'<tr><td style="background:#e5f4ed;padding:6px 10px">{escape(nome_da_dupla(invictos))}</td>'
        f'<td style="background:#eee8f7;padding:6px 10px">{escape(nome_da_dupla(sem_vitorias))}</td>'
        f'<td style="padding:6px 10px">{inteiro_pt(vezes)}</td>'
        f'<td style="font-weight:700;padding:6px 10px">{percentual(vezes)}</td></tr>'
    )

dupla_exemplo = tuple(
    i for i, chave in enumerate(primeira_fase["chave"]) if chave in {"pvision", "liquid"}
)
exemplo_html = ""
if len(dupla_exemplo) == 2:
    vezes_exemplo = contagem_3_0[dupla_exemplo]
    exemplo_html = (
        f'<p><strong>{escape(nome_da_dupla(dupla_exemplo))}</strong> · ambos 3–0 '
        f'em {percentual(vezes_exemplo)} ({inteiro_pt(vezes_exemplo)} simulações)</p>'
    )

display(HTML(f"""
<div style="{estilo_painel}">
    <h3 style="margin:0 0 4px">2. Probabilidades conjuntas por dupla</h3>
    <p style="margin:0 0 10px">{amostra}</p>
    {mapa_de_duplas(contagem_3_0, "3–0", verde)}
    {exemplo_html}
    {mapa_de_duplas(contagem_0_3, "0–3", roxo)}
    <h4 style="margin:18px 0 4px">Duplas 3–0 e 0–3 que mais saíram juntas</h4>
    <div style="overflow-x:auto">
    <table style="{estilo_tabela}">
        <thead><tr><th scope="col">Ambos 3–0</th><th scope="col">Ambos 0–3</th>
        <th scope="col">Simulações</th><th scope="col">Chance conjunta</th></tr></thead>
        <tbody>{"".join(linhas_extremos)}</tbody>
    </table>
    </div>
</div>
"""))

Equipe,PVISION,FaZe Clan,Team Liquid,HEROIC,Ninjas in Pyjamas,Eternal Fire,DENDELE,Wildcard,Imperial Esports,Lynn Vision Gaming,FlyQuest,Chinggis Warriors,9z Globant,M80,GOSH,100 Thieves
PVISION,,"2,675%","2,568%","2,523%","2,690%","1,908%","0,000%","1,133%","1,272%","0,684%","0,778%","0,593%","1,129%","1,104%","0,538%","0,947%"
FaZe Clan,,,"1,782%","2,033%","1,949%","1,577%","0,959%","0,000%","0,835%","0,928%","0,897%","0,861%","0,866%","0,812%","0,564%","0,662%"
Team Liquid,,,,"2,004%","1,981%","1,348%","1,056%","1,054%","1,103%","1,164%","0,950%","1,041%","0,995%","0,934%","0,000%","0,813%"
HEROIC,,,,,"1,986%","1,354%","0,992%","0,790%","0,000%","0,719%","0,674%","0,650%","0,854%","0,804%","0,460%","0,640%"
Ninjas in Pyjamas,,,,,,"1,404%","1,200%","0,952%","1,093%","0,000%","0,674%","0,413%","0,870%","0,000%","0,484%","0,695%"
Eternal Fire,,,,,,,"0,809%","0,797%","0,859%","1,041%","0,000%","0,928%","0,740%","0,633%","0,444%","0,494%"
DENDELE,,,,,,,,"0,726%","0,567%","0,743%","0,668%","0,718%","0,658%","0,451%","0,426%","0,445%"
Wildcard,,,,,,,,,"0,636%","0,771%","0,717%","0,755%","0,658%","0,604%","0,481%","0,533%"
Imperial Esports,,,,,,,,,,"0,777%","0,693%","0,785%","0,631%","0,574%","0,531%","0,488%"
Lynn Vision Gaming,,,,,,,,,,,"0,817%","0,921%","0,000%","0,590%","0,541%","0,617%"


In [20]:
# Gráfico 3 — combinação completa das campanhas dos 16 times
# Cada tupla é um resultado de fase completo de uma única simulação.
contagem_completa = Counter(map(tuple, cenarios.tolist()))
maior_frequencia = max(contagem_completa.values())
perfis_lideres = [
    perfil for perfil, vezes in contagem_completa.items() if vezes == maior_frequencia
]
assert sum(contagem_completa.values()) == repeticoes

paineis = []
for numero, perfil in enumerate(perfis_lideres, start=1):
    linhas = []
    for codigo, placar in enumerate(placares):
        equipes = [escape(nome) for nome, resultado in zip(nomes, perfil, strict=True) if resultado == codigo]
        cor = "#ddf2e7" if codigo < 3 else "#ede5f6"
        cartoes = "".join(
            f'<span style="background:{cor};border-radius:4px;padding:5px 8px;margin:2px 4px 2px 0;'
            f'display:inline-block">{equipe}</span>'
            for equipe in equipes
        )
        linhas.append(
            f'<tr><th scope="row" style="background:{cor};padding:5px 10px">{placar}</th>'
            f'<td style="text-align:left;padding:2px 6px">{cartoes}</td></tr>'
        )
    aberto = " open" if numero == 1 else ""
    paineis.append(f"""
    <details{aberto} style="margin:10px 0">
        <summary style="font-weight:650">Cenário {numero} · {percentual(maior_frequencia)}</summary>
        <table style="{estilo_tabela}">
            <thead><tr><th scope="col">Campanha</th><th scope="col" style="text-align:left">Equipes</th></tr></thead>
            <tbody>{"".join(linhas)}</tbody>
        </table>
    </details>
    """)

topo = (
    "Uma combinação teve a maior frequência."
    if len(perfis_lideres) == 1
    else f"{inteiro_pt(len(perfis_lideres))} combinações empataram na maior frequência."
)
display(HTML(f"""
<div style="{estilo_painel}">
    <h3 style="margin:0 0 4px">3. Combinação completa mais frequente</h3>
    <p style="margin:0 0 10px">{amostra}</p>
    <p>{topo} A maior frequência foi de <strong>{inteiro_pt(maior_frequencia)} ocorrências
        ({percentual(maior_frequencia)})</strong>. Foram observados
        {inteiro_pt(len(contagem_completa))} cenários completos distintos.</p>
    {"".join(paineis)}
</div>
"""))

Campanha,Equipes
3–0,FaZe ClanEternal Fire
3–1,Chinggis WarriorsM80100 Thieves
3–2,PVISIONDENDELEImperial Esports
2–3,Team LiquidHEROICGOSH
1–3,Ninjas in PyjamasLynn Vision Gaming9z Globant
0–3,WildcardFlyQuest
Campanha,Equipes
3–0,Team LiquidHEROIC
3–1,Ninjas in PyjamasChinggis Warriors100 Thieves
3–2,PVISIONFaZe ClanEternal Fire
